# DPO: derive it, implement it, stress it

LLM Training Lab, module 18. Regime PREFERENCE: a LoRA adapter on an SFT'd Qwen3-0.6B-Base, trained with the DPO loss on UltraFeedback preference pairs, against a frozen reference.

Run the cells in order on one T4 (Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2). Steps 1 to 3 also run on a CPU. Code cell numbers match the lab page.

In [ ]:
import os, sys, platform, shutil, subprocess
from importlib.metadata import version, PackageNotFoundError
os.environ["CUDA_VISIBLE_DEVICES"] = "0"            # one GPU, set before torch loads
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
problems = []
if sys.version_info < (3, 11):
    problems.append("Python 3.11 or newer is needed (3.12 recommended). Start a current Colab or Kaggle session.")
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:                                   # a CPU runtime has no nvidia-smi
    smi = []
gpu = smi[0] if smi else ""
if not gpu:
    problems.append("No GPU: steps 1 to 3 still run. For step 4 on, Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2.")
elif "P100" in gpu:
    problems.append("A P100 has no kernels in current PyTorch. Pick a T4.")
cache = os.path.expanduser("~/.cache")
os.makedirs(cache, exist_ok=True)
free_disk = shutil.disk_usage(cache).free / 1e9
if free_disk < 12:
    problems.append("Free at least 12 GB of disk for the model and data downloads (" + str(round(free_disk)) + " GB free).")
try:
    torch_version = version("torch")
except PackageNotFoundError:
    torch_version = "not installed"
print("Python", platform.python_version(), "| GPU", gpu or "none", "| torch", torch_version, "| disk free", round(free_disk), "GB")
for p in problems:
    print("PROBLEM", p)
print("Runtime OK" if not problems else "Read the PROBLEM lines first.")

In [ ]:
# Pinned Hugging Face stack of our recorded run. torch and numpy stay as the runtime ships them.
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 trl==1.17.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 tokenizers==0.23.3 huggingface_hub==2.4.0

In [ ]:
import os, sys, time
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()   # Kaggle, or any Jupyter host
class _Tee:
    def __init__(self, path):
        self.f, self.o = open(path, "a"), sys.stdout
    def write(self, x):
        self.o.write(x); self.f.write(x); self.f.flush()
    def flush(self):
        self.o.flush(); self.f.flush()
    def __getattr__(self, name):
        return getattr(self.o, name)
sys.stdout = _Tee(os.path.join(WORK, "notebook_stdout.txt"))
T_START = time.time()
CELL_TIMES = {}
KERNEL = 'exp'

## Step 1. Derive the DPO loss (NumPy)

Code cell 1 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell01_step1 start", round(_t - T_START, 1))
import numpy as np

def dpo_loss(pw, pl, rw, rl, beta):
    # pw, pl: policy log-probs of the chosen and rejected replies; rw, rl: the same under the frozen reference
    z = beta * ((np.asarray(pw, dtype=float) - rw) - (np.asarray(pl, dtype=float) - rl))
    return float(np.mean(np.logaddexp(0, -z)))

def push(pw, pl, rw, rl, beta):
    # sigmoid(-z): the share of the full gradient a pair still gets, 0.5 at the start and near 0 once it is learned
    z = beta * ((np.asarray(pw, dtype=float) - rw) - (np.asarray(pl, dtype=float) - rl))
    return float(np.mean(1 / (1 + np.exp(z))))

rw, rl = -120.0, -135.0          # reference log-probs of one chosen and one rejected reply, summed over reply tokens
print("policy = reference | loss", round(dpo_loss(rw, rl, rw, rl, 0.1), 4), "| ln 2 =", round(float(np.log(2)), 4))
for name, pw, pl in [("chosen +5, rejected -5", rw + 5, rl - 5),
                     ("chosen -10, rejected -20", rw - 10, rl - 20),
                     ("chosen -40, rejected -50", rw - 40, rl - 50)]:
    print(name, "| loss", round(dpo_loss(pw, pl, rw, rl, 0.1), 4),
          "| implicit rewards", round(0.1 * (pw - rw), 2), "and", round(0.1 * (pl - rl), 2))
for beta in [0.01, 0.1, 0.5]:
    print("beta", beta, "| margin of 10 nats: loss", round(dpo_loss(rw - 10, rl - 20, rw, rl, beta), 4),
          "| push left", round(push(rw - 10, rl - 20, rw, rl, beta), 3))
print("IPO at the start, beta 0.1: (0 - 1 / (2 * 0.1)) squared =", round((0 - 1 / (2 * 0.1)) ** 2, 4))
CELL_TIMES["cell01_step1"] = round(time.time() - _t, 1); print("CELL cell01_step1 end", round(time.time() - T_START, 1))

## Step 2. Score a reply with a masked sequence log-prob

Code cell 2 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell02_step2 start", round(_t - T_START, 1))
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle has 2 GPUs; this lab uses one
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_num_threads(4)
BASE = "Qwen/Qwen3-0.6B-Base"
tok = AutoTokenizer.from_pretrained(BASE)
cpu_model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).eval()

# The chat-control tokens of this base model share one embedding row that was never trained
E = cpu_model.get_input_embeddings().weight
ctrl = [tok.convert_tokens_to_ids(t) for t in ["<|im_start|>", "<|im_end|>", "<think>", "</think>"]]
print("control rows identical:", all(torch.equal(E[ctrl[0]], E[i]) for i in ctrl), "| their norm", round(E[ctrl[0]].norm().item(), 4),
      "| <|endoftext|> norm", round(E[tok.eos_token_id].norm().item(), 4), "| median row norm", round(E.norm(dim=1).median().item(), 4))

NL = chr(10)
TEMPLATE = "### Instruction:" + NL + "{}" + NL + NL + "### Response:" + NL     # plain text, trained tokens only

def encode(prompt, reply):
    # prompt ids, and prompt + reply + <|endoftext|> ids; the prompt must be a prefix of the full row
    p = tok(TEMPLATE.format(prompt)).input_ids
    full = tok(TEMPLATE.format(prompt) + reply).input_ids + [tok.eos_token_id]
    assert full[:len(p)] == p, "the prompt is not a prefix of prompt + reply"
    return p, full

def seq_logps(model, rows, plens, mask_prompt=True):
    # Summed log-prob of each reply. rows: token-id lists; plens: prompt length of each row.
    dev = next(model.parameters()).device
    L = max(len(r) for r in rows)
    ids = torch.full((len(rows), L), tok.pad_token_id, device=dev)
    score = torch.zeros((len(rows), L), dtype=torch.bool, device=dev)
    for i, (r, p) in enumerate(zip(rows, plens)):
        ids[i, :len(r)] = torch.tensor(r, device=dev)
        score[i, (p if mask_prompt else 1):len(r)] = True       # mask from lengths: the pad id is also the EOS id
    att = (torch.arange(L, device=dev)[None] < torch.tensor([len(r) for r in rows], device=dev)[:, None]).long()
    logits = model(input_ids=ids, attention_mask=att).logits[:, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, ids[:, 1:, None]).squeeze(-1)
    return (lp * score[:, 1:]).sum(-1)

p, c = encode("Name a fruit.", "A mango.")
_, r = encode("Name a fruit.", "Fruit is food.")
print("prompt", len(p), "tokens | chosen row", len(c), "| rejected row", len(r), "| pad id = EOS id:", tok.pad_token_id == tok.eos_token_id)
with torch.no_grad():
    masked = seq_logps(cpu_model, [c, r], [len(p)] * 2)
    whole = seq_logps(cpu_model, [c, r], [len(p)] * 2, mask_prompt=False)
print("reply log-probs", [round(x, 3) for x in masked.tolist()], "| with prompt tokens counted", [round(x, 3) for x in whole.tolist()])
print("chosen minus rejected, reply only", round((masked[0] - masked[1]).item(), 4), "| prompt counted", round((whole[0] - whole[1]).item(), 4))
CELL_TIMES["cell02_step2"] = round(time.time() - _t, 1); print("CELL cell02_step2 end", round(time.time() - T_START, 1))

## Step 3. Match DPOTrainer's loss on one batch

Code cell 3 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell03_step3 start", round(_t - T_START, 1))
import gc
from datasets import Dataset, load_dataset
from trl import DPOConfig, DPOTrainer

uf = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
rows = []
for ex in uf:                                    # the first 4 pairs, in file order, whose two rows fit 320 tokens
    c, r = ex["chosen"][-1]["content"].strip(), ex["rejected"][-1]["content"].strip()
    if c != r and max(len(encode(ex["prompt"], c)[1]), len(encode(ex["prompt"], r)[1])) <= 320:
        rows.append({"prompt": TEMPLATE.format(ex["prompt"]), "chosen": c, "rejected": r, "raw": ex["prompt"]})
    if len(rows) == 4:
        break

policy = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32).eval()   # any policy that differs from the reference
args = DPOConfig(output_dir="dpo-check", report_to="none", use_cpu=True, bf16=False, beta=0.1, per_device_train_batch_size=4)
trainer = DPOTrainer(model=policy, ref_model=cpu_model, args=args, processing_class=tok,
                     train_dataset=Dataset.from_list([{k: x[k] for k in ("prompt", "chosen", "rejected")} for x in rows]))
same = all(list(trainer.train_dataset[i]["prompt_ids"]) == encode(x["raw"], x["chosen"])[0] and
           list(trainer.train_dataset[i]["prompt_ids"]) + list(trainer.train_dataset[i]["chosen_ids"]) == encode(x["raw"], x["chosen"])[1]
           for i, x in enumerate(rows))
print("TRL token ids equal yours, EOS included:", same)

batch = next(iter(trainer.get_train_dataloader()))
n = batch["input_ids"].shape[0] // 2                        # chosen rows first, then rejected rows
full = [batch["input_ids"][i, :int(batch["attention_mask"][i].sum())].tolist() for i in range(2 * n)]
plens = [int(batch["completion_mask"][i].argmax()) for i in range(2 * n)]
with torch.no_grad():
    pi, ref = seq_logps(policy, full, plens), seq_logps(cpu_model, full, plens)
    z = 0.1 * ((pi[:n] - ref[:n]) - (pi[n:] - ref[n:]))
    own = -torch.nn.functional.logsigmoid(z).mean().item()
    lib = trainer.compute_loss(trainer.model, dict(batch)).item()
print("batch", list(batch["input_ids"].shape), "| own", round(own, 6), "| TRL", round(lib, 6), "| difference", abs(own - lib))
print("policy minus reference, chosen", [round(x, 2) for x in (pi[:n] - ref[:n]).tolist()],
      "| rejected", [round(x, 2) for x in (pi[n:] - ref[n:]).tolist()])
del trainer, policy
gc.collect()
CELL_TIMES["cell03_step3"] = round(time.time() - _t, 1); print("CELL cell03_step3 end", round(time.time() - T_START, 1))

## Step 4. Build the SFT starting point (RUN_SFT = False downloads ours)

Code cell 4 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell04_step4 start", round(_t - T_START, 1))
import math, random, time
import numpy as np

del cpu_model                          # steps 2 and 3 are done with the CPU copy
gc.collect()
MAXLEN = 512
N_SFT, N_DPO, N_HELD, N_EVAL = 3000, 1000, 300, 200

def fitting_pairs(split, shuffle):
    # UltraFeedback pairs whose chosen and rejected rows both fit MAXLEN tokens; seed-0 shuffle or file order
    ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split=split)
    for ex in (ds.shuffle(seed=0) if shuffle else ds):
        c, r = ex["chosen"][-1]["content"].strip(), ex["rejected"][-1]["content"].strip()
        if c == r:
            continue
        p, ci = encode(ex["prompt"], c)
        ri = encode(ex["prompt"], r)[1]
        if len(ci) <= MAXLEN and len(ri) <= MAXLEN:
            yield {"prompt": ex["prompt"], "chosen": c, "rejected": r, "p": p, "c": ci, "r": ri}

stream = fitting_pairs("train_prefs", shuffle=True)
sft_pairs = [next(stream) for _ in range(N_SFT)]          # SFT on their chosen replies
dpo_pairs = [next(stream) for _ in range(N_DPO)]          # DPO on the next 1,000, never seen in SFT
test = fitting_pairs("test_prefs", shuffle=False)
held = [next(test) for _ in range(N_HELD)]                # held-out pairs for log-prob tracking
eval_prompts = [h["prompt"] for h in held if len(h["p"]) <= 200][:N_EVAL]   # prompts for generation
for name, ps in [("SFT", sft_pairs), ("DPO", dpo_pairs), ("held-out", held)]:
    print(name, len(ps), "pairs | chosen longer", round(float(np.mean([len(x["c"]) > len(x["r"]) for x in ps])), 3),
          "| mean reply tokens, chosen", round(float(np.mean([len(x["c"]) - len(x["p"]) for x in ps]))),
          "rejected", round(float(np.mean([len(x["r"]) - len(x["p"]) for x in ps]))))
print("generation prompts", len(eval_prompts))
CELL_TIMES["cell04_step4"] = round(time.time() - _t, 1); print("CELL cell04_step4 end", round(time.time() - T_START, 1))

In [ ]:
_t = time.time(); print("CELL cell05_step4 start", round(_t - T_START, 1))
from peft import LoraConfig, get_peft_model
from huggingface_hub import snapshot_download
GPU = torch.device("cuda")
RUN_SFT = False          # True trains the SFT model on your T4; False downloads ours from the Hub
runs = {}                # every measured number this notebook prints is also kept here

def lora_on(model, r=16):
    # LoRA on every linear layer of a frozen fp16 model; the adapter weights stay fp32 for the GradScaler
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model.config.use_cache = False
    m = get_peft_model(model, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM"))
    for q in m.parameters():
        if q.requires_grad:
            q.data = q.data.float()
    return m

def plan(n, size, seed, lengths):
    # shuffle, sort each chunk of 64 by length, cut into batches, shuffle the batches: less padding, same data
    rng = random.Random(seed)
    order = list(range(n))
    rng.shuffle(order)
    out = []
    for k in range(0, n, 64):
        chunk = sorted(order[k:k + 64], key=lambda i: -lengths[i])
        out += [chunk[j:j + size] for j in range(0, len(chunk), size)]
    rng.shuffle(out)
    return out

def cosine(step, total, warm):
    return (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))

def train_sft(pairs, lr=2e-4, size=8, micro=4, seed=0):
    torch.manual_seed(seed)
    base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16).to(GPU)
    model = lora_on(base)
    steps = plan(len(pairs), size, seed, [len(x["c"]) for x in pairs])
    opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: cosine(s, len(steps), max(1, len(steps) // 20)))
    scaler = torch.amp.GradScaler("cuda", init_scale=2048)
    torch.cuda.reset_peak_memory_stats()
    t0, hist = time.time(), []
    model.train()
    for step, idx in enumerate(steps):
        opt.zero_grad(set_to_none=True)
        reply_tokens = sum(len(pairs[i]["c"]) - len(pairs[i]["p"]) for i in idx)
        total = 0.0
        for k in range(0, len(idx), micro):
            part = [pairs[i] for i in idx[k:k + micro]]
            with torch.autocast("cuda", dtype=torch.float16):
                lp = seq_logps(model, [x["c"] for x in part], [len(x["p"]) for x in part])
            loss = -lp.sum() / reply_tokens          # mean NLL per reply token over the whole batch
            scaler.scale(loss).backward()
            total += loss.item()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        hist.append(total)
        if (step + 1) % 75 == 0 or step == 0:
            print("SFT step", step + 1, "of", len(steps), "| loss", round(sum(hist[-75:]) / len(hist[-75:]), 4), "| seconds", round(time.time() - t0))
    runs["sft"] = {"steps": len(steps), "sec": round(time.time() - t0, 1), "peakGB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
                   "lossFirst75": round(sum(hist[:75]) / 75, 4), "lossLast75": round(sum(hist[-75:]) / 75, 4),
                   "loss": [round(x, 4) for x in hist]}
    print({k: v for k, v in runs["sft"].items() if k != "loss"})
    merged = model.merge_and_unload()            # fold the adapter into the fp16 weights
    merged.config.use_cache = True
    merged.save_pretrained("sft-model")
    tok.save_pretrained("sft-model")
    del model, merged, base
    gc.collect()
    torch.cuda.empty_cache()
    return "sft-model"

if RUN_SFT:
    SFT_DIR = train_sft(sft_pairs)
else:
    SFT_DIR = os.path.join(snapshot_download("aimlcompanion/llm-training-lab", allow_patterns="lltDPO/sft-s0/*"), "lltDPO", "sft-s0")

def load_sft():
    m = AutoModelForCausalLM.from_pretrained(SFT_DIR, dtype=torch.float16).to(GPU)
    m.config.use_cache = True
    return m

def generate(model, prompts, temperature=0.7, max_new=256, seed=0, batch=50):
    # one sampled reply per prompt, left padding; a reply ends at its first <|endoftext|>
    tok.padding_side, out = "left", []
    model.eval()
    for k in range(0, len(prompts), batch):
        enc = tok([TEMPLATE.format(q) for q in prompts[k:k + batch]], return_tensors="pt", padding=True).to(GPU)
        torch.manual_seed(seed + k)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, do_sample=True, temperature=temperature, top_p=1.0, top_k=0,
                               max_new_tokens=max_new, use_cache=True, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
        for row in g[:, enc["input_ids"].shape[1]:].tolist():
            ended = tok.eos_token_id in row
            n = row.index(tok.eos_token_id) + 1 if ended else len(row)
            out.append({"text": tok.decode(row[:n], skip_special_tokens=True).strip(), "tokens": n, "ended": ended})
    tok.padding_side = "right"
    return out

sft = load_sft()
print(generate(sft, ["Name three fruits that grow in India."], temperature=0.7)[0])
CELL_TIMES["cell05_step4"] = round(time.time() - _t, 1); print("CELL cell05_step4 end", round(time.time() - T_START, 1))

## Step 5. Train DPO by hand (RUN_DPO = False downloads our adapter)

Code cell 6 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell06_step5 start", round(_t - T_START, 1))
import torch.nn.functional as F
from peft import PeftModel
RUN_DPO = False          # True trains DPO on your T4; False downloads our adapter from the Hub

def reference_logps(model, pairs, rows=8):
    # one pass of the frozen SFT model over every pair, so training never holds a reference model
    out = torch.zeros(len(pairs), 2)
    order = sorted(range(len(pairs)), key=lambda i: -len(pairs[i]["c"]) - len(pairs[i]["r"]))
    model.eval()
    for k in range(0, len(order), rows // 2):
        idx = order[k:k + rows // 2]
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            lp = seq_logps(model, [pairs[i]["c"] for i in idx] + [pairs[i]["r"] for i in idx], [len(pairs[i]["p"]) for i in idx] * 2)
        out[idx, 0], out[idx, 1] = lp[:len(idx)].float().cpu(), lp[len(idx):].float().cpu()
    return out

t0 = time.time()
ref_dpo, ref_held = reference_logps(sft, dpo_pairs), reference_logps(sft, held)
runs["refSec"] = round(time.time() - t0, 1)
print("reference log-probs for", len(dpo_pairs) + len(held), "pairs in", runs["refSec"], "s | mean chosen", round(ref_dpo[:, 0].mean().item(), 1),
      "rejected", round(ref_dpo[:, 1].mean().item(), 1))
del sft
gc.collect()
torch.cuda.empty_cache()

def pref_loss(kind, pc, pr, rc, rr, nc, nr, beta, gamma=1.0, kl=None):
    # pc, pr: policy reply log-probs; rc, rr: reference; nc, nr: reply lengths in tokens. One loss per pair.
    if kind == "dpo":
        return -F.logsigmoid(beta * ((pc - rc) - (pr - rr)))
    if kind == "ipo":            # TRL's form: log-ratios averaged per token, squared distance to 1 / (2 beta)
        return ((pc - rc) / nc - (pr - rr) / nr - 1 / (2 * beta)) ** 2
    if kind == "simpo":          # no reference: per-token log-probs and a target margin gamma
        return -F.logsigmoid(beta * (pc / nc - pr / nr) - gamma)
    if kind == "orpo":           # no reference: NLL of the chosen reply plus beta times an odds-ratio loss
        ac, ar = pc / nc, pr / nr
        log_odds = (ac - ar) - (torch.log(-torch.expm1(ac)) - torch.log(-torch.expm1(ar)))
        return -ac - beta * F.logsigmoid(log_odds)
    if kind == "kto":            # chosen and rejected scored on their own against a KL reference point
        return 0.5 * ((1 - torch.sigmoid(beta * ((pc - rc) - kl))) + (1 - torch.sigmoid(beta * (kl - (pr - rr)))))
    raise ValueError(kind)

def kto_kl(model, part):
    # KL estimate from mismatched rows: prompt i with the chosen reply of pair i + 1, policy minus reference, no gradient
    rows = [x["p"] + part[(i + 1) % len(part)]["c"][len(part[(i + 1) % len(part)]["p"]):] for i, x in enumerate(part)]
    plens = [len(x["p"]) for x in part]
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        pol = seq_logps(model, rows, plens)
        with model.disable_adapter():
            ref = seq_logps(model, rows, plens)
    return (pol - ref).mean().clamp(min=0)

def held_stats(model, step):
    # policy minus reference log-prob of every held-out reply, summarised
    lp = torch.zeros(len(held), 2)
    model.eval()
    for k in range(0, len(held), 4):
        part = held[k:k + 4]
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            x = seq_logps(model, [h["c"] for h in part] + [h["r"] for h in part], [len(h["p"]) for h in part] * 2).float().cpu()
        lp[k:k + len(part), 0], lp[k:k + len(part), 1] = x[:len(part)], x[len(part):]
    model.train()
    dc, dr = (lp[:, 0] - ref_held[:, 0]).numpy(), (lp[:, 1] - ref_held[:, 1]).numpy()
    return {"step": step, "chosen": round(float(dc.mean()), 2), "rejected": round(float(dr.mean()), 2),
            "margin": round(float((dc - dr).mean()), 2), "accuracy": round(float((dc > dr).mean()), 3),
            "chosenDown": round(float((dc < 0).mean()), 3), "bothDown": round(float(((dc < 0) & (dr < 0)).mean()), 3), "dc": dc, "dr": dr}

def train_pref(pairs, ref, kind="dpo", beta=0.1, lr=5e-5, seed=0, size=8, micro=2, gamma=1.0, every=25, epochs=1):
    torch.manual_seed(seed)
    model = lora_on(load_sft())
    steps = [b for e in range(epochs) for b in plan(len(pairs), size, seed + 1000 * e, [len(x["c"]) + len(x["r"]) for x in pairs])]
    opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: cosine(s, len(steps), max(1, len(steps) // 10)))
    scaler = torch.amp.GradScaler("cuda", init_scale=2048)
    t0, log, track, cost = time.time(), [], [], {"evalSec": 0.0, "trainPeak": 0, "evalPeak": 0}

    def check(step):
        # held-out log-probs, timed and kept out of the training memory peak
        if step > 0:                             # the peak since the last check is a training peak
            cost["trainPeak"] = max(cost["trainPeak"], torch.cuda.max_memory_allocated())
        torch.cuda.reset_peak_memory_stats()
        te = time.time()
        track.append(held_stats(model, step))
        cost["evalSec"] += time.time() - te
        cost["evalPeak"] = max(cost["evalPeak"], torch.cuda.max_memory_allocated())
        torch.cuda.reset_peak_memory_stats()

    torch.cuda.reset_peak_memory_stats()        # forget the reference pass and anything before this run
    check(0)
    model.train()
    for step, idx in enumerate(steps):
        opt.zero_grad(set_to_none=True)
        row = {"loss": 0.0, "dc": [], "dr": []}
        for k in range(0, len(idx), micro):
            j = idx[k:k + micro]
            part = [pairs[i] for i in j]
            with torch.autocast("cuda", dtype=torch.float16):
                lp = seq_logps(model, [x["c"] for x in part] + [x["r"] for x in part], [len(x["p"]) for x in part] * 2)
            pc, pr = lp[:len(part)].float(), lp[len(part):].float()
            rc, rr = ref[j, 0].to(GPU), ref[j, 1].to(GPU)
            nc = torch.tensor([len(x["c"]) - len(x["p"]) for x in part], device=GPU)
            nr = torch.tensor([len(x["r"]) - len(x["p"]) for x in part], device=GPU)
            kl = kto_kl(model, part) if kind == "kto" else None
            loss = pref_loss(kind, pc, pr, rc, rr, nc, nr, beta, gamma, kl).sum() / len(idx)
            scaler.scale(loss).backward()
            row["loss"] += loss.item()
            row["dc"] += (pc - rc).tolist()
            row["dr"] += (pr - rr).tolist()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        log.append({"loss": round(row["loss"], 4), "chosen": round(float(np.mean(row["dc"])), 3), "rejected": round(float(np.mean(row["dr"])), 3)})
        if (step + 1) % every == 0:
            check(step + 1)
            h = track[-1]
            print(kind, "beta", beta, "| step", step + 1, "of", len(steps), "| train loss", round(float(np.mean([x["loss"] for x in log[-every:]])), 4),
                  "| held-out chosen", h["chosen"], "rejected", h["rejected"], "accuracy", h["accuracy"], "| seconds", round(time.time() - t0))
    if len(steps) % every:                       # the last step was not checked yet
        check(len(steps))
    info = {"kind": kind, "beta": beta, "lr": lr, "seed": seed, "pairs": len(pairs), "epochs": epochs, "steps": len(steps), "sec": round(time.time() - t0, 1),
            "trainSec": round(time.time() - t0 - cost["evalSec"], 1), "trainPeakGB": round(cost["trainPeak"] / 1e9, 2),
            "evalPeakGB": round(cost["evalPeak"] / 1e9, 2), "log": log,
            "held": [{k: v for k, v in h.items() if k not in ("dc", "dr")} for h in track]}
    return model, info, track[-1]

if RUN_DPO:
    dpo, info, final = train_pref(dpo_pairs, ref_dpo, "dpo", beta=0.1)
    dpo.save_pretrained("dpo-b0.1-s0")
    runs["dpo-b0.1-s0"] = info
    print("training peak GB", info["trainPeakGB"], "| held-out check peak GB", info["evalPeakGB"], "| seconds", info["sec"],
          "of which held-out checks", round(info["sec"] - info["trainSec"], 1))
else:
    path = snapshot_download("aimlcompanion/llm-training-lab", allow_patterns="lltDPO/dpo-b0.1-s0/*")
    dpo = PeftModel.from_pretrained(load_sft(), os.path.join(path, "lltDPO", "dpo-b0.1-s0"))
    final = held_stats(dpo, 125)
print({k: v for k, v in final.items() if k not in ("dc", "dr")})
CELL_TIMES["cell06_step5"] = round(time.time() - _t, 1); print("CELL cell06_step5 end", round(time.time() - T_START, 1))

## Step 6. Read the log-probs, not only the loss

Code cell 7 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell07_step6 start", round(_t - T_START, 1))
dc, dr = final["dc"], final["dr"]
nc = np.array([len(h["c"]) - len(h["p"]) for h in held])
nr = np.array([len(h["r"]) - len(h["p"]) for h in held])
print("held-out pairs", len(held), "| chosen log-prob fell in", round(float((dc < 0).mean()), 3), "| rejected fell in", round(float((dr < 0).mean()), 3),
      "| both fell in", round(float(((dc < 0) & (dr < 0)).mean()), 3), "| margin grew in", round(float((dc > dr).mean()), 3))
print("mean change per reply token, chosen", round(float((dc / nc).mean()), 4), "| rejected", round(float((dr / nr).mean()), 4))
print("correlation between the chosen and rejected change of a pair", round(float(np.corrcoef(dc, dr)[0, 1]), 3))
for h in runs.get("dpo-b0.1-s0", {}).get("held", []):
    print("step", h["step"], "| chosen", h["chosen"], "| rejected", h["rejected"], "| margin", h["margin"], "| accuracy", h["accuracy"],
          "| chosen fell", h["chosenDown"])
CELL_TIMES["cell07_step6"] = round(time.time() - _t, 1); print("CELL cell07_step6 end", round(time.time() - T_START, 1))

## Step 7. Judge the policy with a gold reward model

Code cell 8 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell08_step7 start", round(_t - T_START, 1))
from transformers import AutoModelForSequenceClassification
GOLD = "Skywork/Skywork-Reward-V2-Qwen3-1.7B"         # the gold judge, also the labeller in step 11
JUDGE2 = "Skywork/Skywork-Reward-V2-Llama-3.2-1B"      # a second judge on another base model and tokenizer

def score_seqs(model, seqs, pad_id, budget=12288):
    # one reward per token-id list: batches of similar length, at most 32 rows or 12,288 tokens, right padding
    order, out, k = sorted(range(len(seqs)), key=lambda i: -len(seqs[i])), [None] * len(seqs), 0
    while k < len(order):
        idx = order[k:k + max(1, min(32, budget // len(seqs[order[k]])))]
        ids = torch.full((len(idx), len(seqs[idx[0]])), pad_id)
        att = torch.zeros_like(ids)
        for i, j in enumerate(idx):
            ids[i, :len(seqs[j])], att[i, :len(seqs[j])] = torch.tensor(seqs[j]), 1
        with torch.no_grad():
            r = model(input_ids=ids.to(GPU), attention_mask=att.to(GPU)).logits.squeeze(-1).float().tolist()
        for i, j in enumerate(idx):
            out[j] = r[i]
        k += len(idx)
    return np.array(out)

def gold_scores(sets, judge=GOLD):
    # sets: {name: list of (prompt, reply)}. Loads the judge once, scores in its own chat format, frees it.
    gtok = AutoTokenizer.from_pretrained(judge)
    if gtok.pad_token is None:
        gtok.pad_token = gtok.eos_token
    gold_rm = AutoModelForSequenceClassification.from_pretrained(judge, num_labels=1, dtype=torch.float16).to(GPU).eval()
    gold_rm.config.pad_token_id = gtok.pad_token_id
    out = {}
    for name, items in sets.items():
        seqs = []
        for q, a in items:
            s = gtok.apply_chat_template([{"role": "user", "content": q}, {"role": "assistant", "content": a}], tokenize=False)
            seqs.append(gtok(s, add_special_tokens=False).input_ids[:4096])     # the template already holds any BOS
        out[name] = score_seqs(gold_rm, seqs, gtok.pad_token_id)
    del gold_rm
    gc.collect()
    torch.cuda.empty_cache()
    return out

def paired(a, b, seed=0):
    # win rate of a over b on the same prompts (a tie counts half) and the mean gold difference, 95% bootstrap intervals
    w = 0.5 + 0.5 * np.sign(a - b)
    boot = np.random.default_rng(seed).integers(0, len(w), (10000, len(w)))
    wl, wh = np.percentile(w[boot].mean(1), [2.5, 97.5])
    dl, dh = np.percentile((a - b)[boot].mean(1), [2.5, 97.5])
    return {"win": round(float(w.mean()), 3), "winCI": [round(float(wl), 3), round(float(wh), 3)],
            "diff": round(float((a - b).mean()), 3), "diffCI": [round(float(dl), 3), round(float(dh), 3)]}

gens = {"dpo-b0.1-s0": generate(dpo, eval_prompts)}
with dpo.disable_adapter():                       # adapter off = the SFT model, same seeds
    gens["sft"] = generate(dpo, eval_prompts)
del dpo
gc.collect()
torch.cuda.empty_cache()
gold = gold_scores({k: list(zip(eval_prompts, [g["text"] for g in v])) for k, v in gens.items()})
judge2 = gold_scores({k: list(zip(eval_prompts, [g["text"] for g in v])) for k, v in gens.items()}, JUDGE2)
for k in ["sft", "dpo-b0.1-s0"]:
    print(k, "| gold mean", round(float(gold[k].mean()), 3), "| reply tokens", round(float(np.mean([g["tokens"] for g in gens[k]])), 1),
          "| ended", round(float(np.mean([g["ended"] for g in gens[k]])), 3))
runs["judge"], runs["judge2"] = paired(gold["dpo-b0.1-s0"], gold["sft"]), paired(judge2["dpo-b0.1-s0"], judge2["sft"])
print("DPO against SFT on", len(eval_prompts), "prompts, gold judge:", runs["judge"])
print("second judge:", runs["judge2"])
CELL_TIMES["cell08_step7"] = round(time.time() - _t, 1); print("CELL cell08_step7 end", round(time.time() - T_START, 1))

## Step 8. Run the same training with DPOTrainer

Code cell 9 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell09_step8 start", round(_t - T_START, 1))
trl_rows = Dataset.from_list([{"prompt": TEMPLATE.format(x["prompt"]), "chosen": x["chosen"], "rejected": x["rejected"]}
                              for x in dpo_pairs[:320]])
cfg = DPOConfig(output_dir="dpo-trl", report_to="none", fp16=True, bf16=False, beta=0.1, learning_rate=5e-5,
                lr_scheduler_type="cosine", warmup_steps=4, per_device_train_batch_size=2, gradient_accumulation_steps=4,
                max_steps=40, max_length=MAXLEN, gradient_checkpointing=True, precompute_ref_log_probs=True,
                logging_steps=10, save_strategy="no", seed=0)
trainer = DPOTrainer(model=load_sft(), args=cfg, processing_class=tok, train_dataset=trl_rows,
                     peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM"))
kinds = {q.dtype for q in trainer.model.parameters() if q.requires_grad}
for q in trainer.model.parameters():
    if q.requires_grad and q.dtype != torch.float32:
        q.data = q.data.float()                   # fp16 adapter weights break the GradScaler; make them fp32
print("trainable dtypes after the trainer was built", kinds, "| now", {q.dtype for q in trainer.model.parameters() if q.requires_grad})
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
runs["trl"] = {"steps": 40, "sec": round(time.time() - t0, 1), "sPerStep": round((time.time() - t0) / 40, 2),
               "peakGB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
               "log": [{k: round(v, 4) for k, v in h.items() if isinstance(v, float)} for h in trainer.state.log_history]}
print("DPOTrainer, 40 steps of 8 pairs:", runs["trl"]["sPerStep"], "s per step | peak GB", runs["trl"]["peakGB"])
for h in runs["trl"]["log"][:4]:
    print(h)
del trainer
gc.collect()
torch.cuda.empty_cache()
CELL_TIMES["cell09_step8"] = round(time.time() - _t, 1); print("CELL cell09_step8 end", round(time.time() - T_START, 1))

## Step 9. Sweep beta (RUN_EXPERIMENT = True trains three more models)

Code cell 10 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell10_step9 start", round(_t - T_START, 1))
RUN_EXPERIMENT = False   # True trains three more DPO models on your T4; our runs are in the table of this step

def experiment(arms, pairs, ref):
    # arms: (name, loss, beta, seed, extra). Train each arm, sample from it, free it; then score every arm against SFT.
    for name, kind, beta, seed, extra in arms:
        model, info, last = train_pref(pairs, ref, kind, beta=beta, seed=seed, **extra)
        gens[name] = generate(model, eval_prompts)
        info["final"] = {k: v for k, v in last.items() if k not in ("dc", "dr")}
        info["perPair"] = {"chosen": np.round(last["dc"], 2).tolist(), "rejected": np.round(last["dr"], 2).tolist()}
        runs[name] = info
        del model
        gc.collect()
        torch.cuda.empty_cache()
    sets = {a[0]: list(zip(eval_prompts, [g["text"] for g in gens[a[0]]])) for a in arms}
    scores, scores2 = gold_scores(sets), gold_scores(sets, JUDGE2)
    for name, kind, beta, seed, extra in arms:
        r, gold[name], judge2[name] = runs[name], scores[name], scores2[name]
        r["gold"], r["vsSFT"] = round(float(scores[name].mean()), 3), paired(scores[name], gold["sft"])
        r["vsSFT2"] = paired(scores2[name], judge2["sft"])
        r["tokens"] = round(float(np.mean([g["tokens"] for g in gens[name]])), 1)
        r["ended"] = round(float(np.mean([g["ended"] for g in gens[name]])), 3)
        print(name, "| held-out chosen", r["final"]["chosen"], "rejected", r["final"]["rejected"], "accuracy", r["final"]["accuracy"],
              "| win vs SFT", r["vsSFT"]["win"], r["vsSFT"]["winCI"], "| second judge", r["vsSFT2"]["win"], r["vsSFT2"]["winCI"], "| reply tokens", r["tokens"], "| ended", r["ended"])

if RUN_EXPERIMENT:
    experiment([("dpo-b0.01-s0", "dpo", 0.01, 0, {}), ("dpo-b0.5-s0", "dpo", 0.5, 0, {}), ("dpo-b0.1-s1", "dpo", 0.1, 1, {})],
               dpo_pairs, ref_dpo)
CELL_TIMES["cell10_step9"] = round(time.time() - _t, 1); print("CELL cell10_step9 end", round(time.time() - T_START, 1))

## Step 10. Compare IPO, SimPO, ORPO and KTO (RUN_EXPERIMENT = True trains four models)

Code cell 11 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell11_step10 start", round(_t - T_START, 1))
RUN_EXPERIMENT = True   # True trains four more models on your T4; our runs are in the table of this step

if RUN_EXPERIMENT:
    experiment([("ipo-s0", "ipo", 5.0, 0, {"every": 125}),                  # TRL: DPOConfig(loss_type="ipo", beta=5.0)
                ("simpo-s0", "simpo", 2.0, 0, {"gamma": 1.0, "every": 125}),  # TRL: CPOConfig(loss_type="simpo", cpo_alpha=0), experimental
                ("orpo-s0", "orpo", 0.1, 0, {"every": 125}),                # TRL: ORPOConfig(beta=0.1), experimental
                ("kto-s0", "kto", 0.1, 0, {"every": 125})],                 # TRL: KTOConfig(beta=0.1), stable
               dpo_pairs, ref_dpo)
CELL_TIMES["cell11_step10"] = round(time.time() - _t, 1); print("CELL cell11_step10 end", round(time.time() - T_START, 1))

## Step 11. Pairs from your own model, labelled by two reward models (RUN_EXPERIMENT = True)

Code cell 12 on the lab page.

In [ ]:
_t = time.time(); print("CELL cell12_step11 start", round(_t - T_START, 1))
RUN_EXPERIMENT = True   # True samples 2,000 replies from the SFT model, labels them twice and trains three DPO models

def proxy_scores(sets):
    # our 360M reward model from lltRewardModels, in its own chat template, read at the last real token
    path = os.path.join(snapshot_download("aimlcompanion/llm-training-lab", allow_patterns="lltRewardModels/rm-s0/*"), "lltRewardModels", "rm-s0")
    rtok = AutoTokenizer.from_pretrained(path)
    rm = AutoModelForSequenceClassification.from_pretrained(path, dtype=torch.float16).to(GPU).eval()
    out = {name: score_seqs(rm, [list(rtok.apply_chat_template([{"role": "user", "content": q}, {"role": "assistant", "content": a}])["input_ids"])[:1024]
                                 for q, a in items], rtok.pad_token_id) for name, items in sets.items()}
    del rm
    gc.collect()
    torch.cuda.empty_cache()
    return out

def make_pair(q, x, y, sx, sy):
    # the higher-scored reply becomes chosen; None unless both ended, differ, are scored differently and fit MAXLEN
    if not (x["ended"] and y["ended"]) or x["text"] == y["text"] or sx == sy:
        return None
    win, lose = (x, y) if sx > sy else (y, x)
    p, ci = encode(q, win["text"])
    ri = encode(q, lose["text"])[1]
    if len(ci) > MAXLEN or len(ri) > MAXLEN:
        return None
    return {"prompt": q, "chosen": win["text"], "rejected": lose["text"], "p": p, "c": ci, "r": ri}

if RUN_EXPERIMENT:
    prompts = [x["prompt"] for x in dpo_pairs]
    pol, t0 = load_sft(), time.time()
    a, b = generate(pol, prompts, temperature=1.0, seed=0, batch=64), generate(pol, prompts, temperature=1.0, seed=1, batch=64)
    runs["onpolicySampleSec"] = round(time.time() - t0, 1)
    on = {"a": list(zip(prompts, [x["text"] for x in a])), "b": list(zip(prompts, [x["text"] for x in b]))}
    off = {"a": [(x["prompt"], x["chosen"]) for x in dpo_pairs], "b": [(x["prompt"], x["rejected"]) for x in dpo_pairs]}
    g = gold_scores({"on_a": on["a"], "on_b": on["b"], "off_a": off["a"], "off_b": off["b"]})
    pr = proxy_scores(on)
    # keep a prompt only when both labellers give it an on-policy pair, so all three arms use the same prompts and size
    keep, sets = [], {"onpolicy-gold-s0": [], "onpolicy-proxy-s0": [], "offline-gold-s0": []}
    for i, q in enumerate(prompts):
        pg = make_pair(q, a[i], b[i], g["on_a"][i], g["on_b"][i])
        pp = make_pair(q, a[i], b[i], pr["a"][i], pr["b"][i])
        if pg is None or pp is None or g["off_a"][i] == g["off_b"][i]:
            continue
        keep.append(i)
        sets["onpolicy-gold-s0"].append(pg)
        sets["onpolicy-proxy-s0"].append(pp)
        x = dpo_pairs[i]                       # the same prompt's UltraFeedback replies, relabelled by the gold model
        sets["offline-gold-s0"].append(dict(x) if g["off_a"][i] > g["off_b"][i] else
                                       dict(x, chosen=x["rejected"], rejected=x["chosen"], c=x["r"], r=x["c"]))
    longer = lambda ps: round(float(np.mean([len(x["c"]) > len(x["r"]) for x in ps])), 3)
    runs["onpolicy-labels"] = {"prompts": len(prompts), "kept": len(keep), "ended": round(float(np.mean([x["ended"] for x in a + b])), 3),
                               "goldAgreesProxy": round(float(np.mean([(g["on_a"][i] > g["on_b"][i]) == (pr["a"][i] > pr["b"][i]) for i in keep])), 3),
                               "goldKeepsUltraFeedbackLabel": round(float(np.mean([g["off_a"][i] > g["off_b"][i] for i in keep])), 3),
                               "chosenLonger": {k: longer(v) for k, v in sets.items()}}
    print(runs["onpolicy-labels"], "| sampling took", runs["onpolicySampleSec"], "s")
    epochs = max(1, round(len(dpo_pairs) / max(1, len(keep))))       # about as many steps as the 1,000-pair runs
    for name, ps in sets.items():
        # held-out checks only at the start and the end of each arm
        experiment([(name, "dpo", 0.1, 0, {"every": 10 ** 6, "epochs": epochs})], ps, reference_logps(pol, ps))
    del pol
    gc.collect()
    torch.cuda.empty_cache()
CELL_TIMES["cell12_step11"] = round(time.time() - _t, 1); print("CELL cell12_step11 end", round(time.time() - T_START, 1))

In [ ]:
import json, torch, transformers, trl, peft, datasets, platform
def _clean(o):
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items() if k not in ("dc", "dr")}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if hasattr(o, "tolist"):
        return o.tolist()
    return o
rec = {"kernel": KERNEL, "gpu": torch.cuda.get_device_name(0), "env": {"python": platform.python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda, "transformers": transformers.__version__, "trl": trl.__version__, "peft": peft.__version__,
       "datasets": datasets.__version__}, "cellTimes": CELL_TIMES, "runs": _clean(runs), "totalSec": round(time.time() - T_START, 1)}
for name in ("gens", "gold", "judge2"):
    if name in globals():
        rec[name] = _clean(globals()[name])
if "final" in globals():
    rec["finalHeld"] = {"dc": final["dc"].tolist(), "dr": final["dr"].tolist()}
json.dump(rec, open(os.path.join(WORK, "llt_dpo_" + KERNEL + ".json"), "w"), indent=1)
print("RECORD written", rec["totalSec"])